<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-15-run-the-meridian-assistant-like-a-system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 15 (graded) — Run the Meridian assistant like a system
**Course 2: Generative AI and LLMs with Python — Chapter 15: Deploy, monitor, detect drift**

**A month after the compliance assistant shipped — Leo Farkas:** "It was accurate in
testing. Now the rulebook changed, one prompt tweak broke JSON output for a week, and our
API bill tripled."

**What you'll submit:** the assistant deployed with tracing + cost logging, the eval suite
as a release gate blocking a bad change, both corpus and query drift caught by monitors, and
a monitoring plan + runbook.

In [ ]:
!pip install -q sentence-transformers faiss-cpu

## 1. Rebuild the Chapter 8/9 pipeline, compactly, with tracing

In [ ]:
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

documents = [
    ('kyc-001', 'Customer Identification Program', 'A government-issued photo ID and proof of address are required for every new account. Accounts may not transact until identity verification is complete.'),
    ('aml-002', 'Suspicious Activity Reporting', 'A Suspicious Activity Report is filed for transactions meeting regulatory thresholds within 24 hours of detection.'),
    ('ege-003', 'Reg E Error Resolution', 'Customers have 60 days to report an unauthorized electronic transfer. Liability is capped at $50 if reported within 2 business days.'),
    ('od-004', 'Overdraft Protection Policy', 'Overdraft fees are capped at 3 per day. Accounts overdrawn more than 60 days are referred to collections.'),
]

embedder = SentenceTransformer('all-MiniLM-L6-v2')

class Corpus:
    def __init__(self, docs):
        self.docs = docs
        self.embeddings = embedder.encode([d[2] for d in docs], normalize_embeddings=True)

    def retrieve(self, query, top_k=1):
        q = embedder.encode([query], normalize_embeddings=True)
        sims = (self.embeddings @ q.T).ravel()
        idx = np.argsort(-sims)[:top_k]
        return [{'title': self.docs[i][1], 'text': self.docs[i][2], 'score': float(sims[i])} for i in idx]

corpus = Corpus(documents)

class TracedAssistant:
    """Every request logs: query, retrieved doc(s), retrieval score, response, latency, and
    an estimated cost — the observability contract from the chapter."""
    def __init__(self, corpus, prompt_version='v1'):
        self.corpus = corpus
        self.prompt_version = prompt_version
        self.trace_log = []

    def answer(self, query, simulate_broken_json=False):
        t0 = time.perf_counter()
        retrieved = self.corpus.retrieve(query)
        best = retrieved[0]
        # a stand-in generator (deterministic) — the point of this lab is the ops layer
        # around generation, not generation quality itself, which Ch 8/9 already covered
        response = ('{malformed json' if simulate_broken_json else
                     f"Per {best['title']}: {best['text']}")
        latency = time.perf_counter() - t0
        est_tokens = len(query.split()) + len(response.split())
        self.trace_log.append({
            'query': query, 'retrieved_title': best['title'], 'retrieval_score': best['score'],
            'response': response, 'latency_s': latency, 'est_tokens': est_tokens,
            'est_cost_usd': est_tokens / 1000 * 0.0002, 'prompt_version': self.prompt_version,
        })
        return response

assistant = TracedAssistant(corpus)
assistant.answer('How long do I have to report a fraudulent transfer?')
pd.DataFrame(assistant.trace_log)

## 2. The eval suite as a release gate

In [ ]:
import json

eval_set = [
    ('How long do I have to report a fraudulent transfer?', 'ege-003'),
    ('What ID is required to open an account?', 'kyc-001'),
    ('How many overdraft fees can be charged per day?', 'od-004'),
]

def release_gate(assistant_fn, eval_set, require_json=False):
    """A minimal release gate: every eval question must retrieve the right document, and (if
    required) the response must be valid JSON. Returns (passed, failures)."""
    failures = []
    for query, expected_doc_id in eval_set:
        retrieved = corpus.retrieve(query)
        if retrieved[0]['title'] != [d[1] for d in documents if d[0] == expected_doc_id][0]:
            failures.append((query, 'wrong document retrieved'))
        response = assistant_fn(query)
        if require_json:
            try:
                json.loads(response)
            except json.JSONDecodeError:
                failures.append((query, 'response is not valid JSON'))
    return len(failures) == 0, failures

# a GOOD change: passes
good_assistant = TracedAssistant(corpus, prompt_version='v2-good')
passed, failures = release_gate(lambda q: good_assistant.answer(q), eval_set, require_json=False)
print(f'Release gate (good change): {"PASSED" if passed else "FAILED"}  failures={failures}')

# a BAD change: the "prompt tweak that broke JSON output" from Leo's story
bad_assistant = TracedAssistant(corpus, prompt_version='v3-broken-json')
passed, failures = release_gate(lambda q: bad_assistant.answer(q, simulate_broken_json=True), eval_set, require_json=True)
print(f'Release gate (bad change):  {"PASSED" if passed else "FAILED"}  failures={failures}')
assert not passed, 'The gate should have blocked the broken-JSON change.'
print('\nThe gate correctly blocked the change that would have broken production for a week.')

## 3. Simulate corpus drift and query drift

In [ ]:
# CORPUS DRIFT: the rulebook changed but the index wasn't refreshed
stale_corpus = Corpus(documents)  # the OLD index
updated_documents = documents.copy()
updated_documents[2] = ('ege-003', 'Reg E Error Resolution (updated)',
                          'Customers now have 90 days to report an unauthorized electronic transfer, up from the previous 60-day window. Liability remains capped at $50 if reported within 2 business days.')

stale_answer = TracedAssistant(stale_corpus).answer('How long do I have to report a fraudulent transfer?')
print('Stale-index answer (should mention the OLD 60-day window):', stale_answer)
print('This is exactly why a scheduled re-indexing job + a "corpus checksum changed" alert matters.')

In [ ]:
# QUERY DRIFT: users start asking about something new the corpus doesn't cover
historical_queries = [q for q, _ in eval_set] * 10
new_wave_queries = [
    'What are the new cryptocurrency custody rules?',
    'How does the bank handle buy-now-pay-later disputes?',
    'What is the policy on AI-generated loan decisions?',
] * 10

def retrieval_hit_rate(queries, threshold=0.35):
    hits = sum(1 for q in queries if corpus.retrieve(q)[0]['score'] >= threshold)
    return hits / len(queries)

hist_rate = retrieval_hit_rate(historical_queries)
new_rate = retrieval_hit_rate(new_wave_queries)
print(f'Retrieval hit-rate on historical queries: {hist_rate:.0%}')
print(f'Retrieval hit-rate on the new query wave:  {new_rate:.0%}')
print('A sustained drop like this is the query-drift signal — the corpus needs new documents,')
print('not a model change.')
assert new_rate < hist_rate, 'Expected the new topic wave to have a lower retrieval hit-rate.'

## 4. Re-indexing DAG sketch (Airflow-style pseudocode)

In [ ]:
reindex_dag_sketch = '''
# Airflow-style pseudocode — not runnable here, documents the trigger + steps
with DAG("meridian_reindex", schedule="@daily") as dag:
    check_corpus_checksum = PythonOperator(task_id="check_corpus_checksum", python_callable=diff_source_docs)
    check_retrieval_hit_rate = PythonOperator(task_id="check_hit_rate", python_callable=compute_hit_rate_on_recent_queries)
    reembed_changed_docs = PythonOperator(task_id="reembed", python_callable=embed_and_upsert)
    notify_compliance_team = PythonOperator(task_id="notify", python_callable=send_slack_alert)

    [check_corpus_checksum, check_retrieval_hit_rate] >> reembed_changed_docs >> notify_compliance_team
'''
print(reindex_dag_sketch)

## 5. Monitoring plan + runbook (fill in)
- **What to monitor:** cost per request, retrieval hit-rate, release-gate pass/fail, corpus
  checksum.
- **Alert thresholds (fill in):** _your numbers here_
- **Runbook — when the hit-rate monitor fires (fill in):** who is paged, what do they check
  first, when do they trigger a re-index vs. escalate to compliance?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 15: Deploy, monitor, detect drift*